This notebook is a tool that scrapes images from the lichen portal in order to generate training data.



In [ ]:
from utils.notebook_setup import setup_project
setup_project()
# SETUP
I_KNOW_WHAT_I_AM_DOING = True
I_KNOW_THIS_IS_AN_INTERNAL_TOOL = True

csv_path = 'data/nov_4_24foliicolousverification.csv'
num_rows_to_import = 1 # Do not set this very high please
randomize_columns = True # highly recommended

In [ ]:
import pandas as pd
import os

from herbarium_processor.config import ROOT_DIR

if not I_KNOW_THIS_IS_AN_INTERNAL_TOOL or not I_KNOW_WHAT_I_AM_DOING:
    raise RuntimeError("i stop now")

# Load the CSV
df = pd.read_csv(ROOT_DIR / csv_path, encoding='ISO-8859-1')

# Filter to rows whose image does not exist
def is_missing_image(row):
    return not os.path.exists(os.path.join(ROOT_DIR / 'data/img', f"{row['id']}.jpg"))

df['id'] = df['id'].astype(str)
df_missing = df[df.apply(is_missing_image, axis=1)]

if randomize_columns:
    df_missing = df_missing.sample(frac=1).reset_index(drop=True)

# Limit to desired number of rows
df_pending = df_missing.head(num_rows_to_import)

# ✅ pending_ids now matches the CSV row content 1:1
pending_ids = df_pending['id'].tolist()


In [ ]:
import requests
from bs4 import BeautifulSoup
from PIL import Image
from io import BytesIO
import os

from herbarium_processor.config import ROOT_DIR

# Define image output path
output_dir = ROOT_DIR / "data/img"
os.makedirs(output_dir, exist_ok=True)

# Image normalization settings
MAX_WIDTH = 1600
DPI = 300

for occid in pending_ids:
    try:
        url = f"https://lichenportal.org/portal/collections/individual/index.php?occid={occid}"

        # Request the specimen page
        response = requests.get(url)
        response.raise_for_status()

        # Parse the HTML for image URL
        soup = BeautifulSoup(response.text, 'html.parser')
        thumb_div = soup.find(id='thumbnail-div')

        if thumb_div and thumb_div.a:
            href = thumb_div.a['href']
            absolute_url = href if href.startswith('http') else f"https://lichenportal.org{href}"

            print("Image URL:", absolute_url)

            # Download the image content
            img_response = requests.get(absolute_url)
            img_response.raise_for_status()

            # Open and normalize the image
            img = Image.open(BytesIO(img_response.content)).convert("RGB")

            # Resize to max width (preserve aspect ratio)
            if img.width > MAX_WIDTH:
                new_height = int((MAX_WIDTH / img.width) * img.height)
                img = img.resize((MAX_WIDTH, new_height), Image.LANCZOS)

            # Save as JPEG with consistent DPI
            output_path = os.path.join(output_dir, f"{occid}.jpg")
            img.save(output_path, format="JPEG", dpi=(DPI, DPI), quality=95)

            print(f"Saved image as {occid}.jpg")

        else:
            print("Image URL not found for id " + occid)

    except Exception as e:
        print(f"[{occid}] Error fetching or processing image: {e}")
